# ActionPulse diarization worker on a free GPU (Kaggle or Colab)

Runs `diarization-worker/app.py` (faster-whisper + pyannote community-1) on the notebook's free GPU and
exposes it through a free Cloudflare quick tunnel. Then paste the printed URL into Render as `DIARIZATION_WORKER_URL`.

**Kaggle:** Settings → *Accelerator* = **GPU T4 x2** (or P100), *Internet* = **On** (needs a phone-verified account).
Add-ons → *Secrets*: `HF_TOKEN` and `WORKER_SECRET`, both attached to this notebook.

**Colab:** Runtime → *Change runtime type* → **T4 GPU**. 🔑 *Secrets* panel: `HF_TOKEN` and `WORKER_SECRET`, notebook access on.

Keep the notebook running while you use the app. Kaggle sessions last up to 12 h (30 GPU h/week);
Colab disconnects sooner when idle. The tunnel URL changes on every run, so update Render each time.

In [ ]:
# 1. Install (about 2-4 min). pyannote.audio may upgrade torch; that's fine.
!pip install -q faster-whisper "pyannote.audio>=4.0.0" fastapi "uvicorn[standard]" python-multipart
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

In [ ]:
# 2. Fetch the worker + cloudflared
BRANCH = "claude/blissful-tesla-oybu12"  # switch to "main" once the PR is merged
!wget -q -O app.py https://raw.githubusercontent.com/Neel7780/AI-Powered-Meeting-to-Action-Converter/$BRANCH/diarization-worker/app.py
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Secrets (Kaggle Secrets or Colab userdata)
import os
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        from google.colab import userdata
        return userdata.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.environ["WORKER_SECRET"] = secret("WORKER_SECRET")
os.environ["WHISPER_MODEL"] = "small"  # "medium" is more accurate and still fits a T4
print("secrets loaded")

In [ ]:
# 4. Start the worker and warm it up with 3 s of silence (first load downloads about 1 GB of models)
import subprocess, time, requests
log = open("worker.log", "w")
worker = subprocess.Popen(["python", "app.py"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        requests.get("http://localhost:7860/"); break
    except Exception:
        time.sleep(2)

!ffmpeg -loglevel error -y -f lavfi -i anullsrc=r=16000:cl=mono -t 3 silence.wav
r = requests.post("http://localhost:7860/diarize", headers={"X-Worker-Secret": os.environ["WORKER_SECRET"]},
                  files={"audio": open("silence.wav", "rb")}, data={"num_speakers": "1"}, timeout=900)
print(r.status_code, r.text[:300])
print(requests.get("http://localhost:7860/").json())  # should say "device": "cuda"
# If this failed, the reason is in worker.log (usually: model terms not accepted, or bad HF_TOKEN)

In [ ]:
# 5. Public HTTPS URL via Cloudflare quick tunnel (free, no account)
import re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:7860"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\nDIARIZATION_WORKER_URL =", m.group(0))
        print("Set it (and DIARIZATION_WORKER_SECRET = your WORKER_SECRET) on Render, then redeploy.")
        break

In [ ]:
# 6. Keep alive + show worker logs (stop this cell to shut down)
import time
while worker.poll() is None:
    time.sleep(60)
    print(open("worker.log").read()[-1500:], flush=True)
print("worker exited:", worker.returncode)